# Camada GOLD - Analise base de dados "Viagens a Serviço do Portal da Transparência do Governo Federal"

Este notebook le a camada **SILVER** (ja limpa e tipada) e responde 3 perguntas
de negocio. Para cada uma: a consulta SQL, a tabela com o resultado e um grafico.

**Perguntas a responder:**

1. Os 5 órgãos com maior custo total?
2. Qual o meio de transporte mais usado nos trechos?
3. Qual UF de destino aparece em mais trechos?


| Pergunta                                              | Gráfico            |                                                                 
| ----------------------------------------------------- | ------------------ | 
| **Os 5 órgãos com maior custo total?**                | Barras horizontais | 
| **Qual UF de destino aparece em mais trechos?**       | Gráfico de Rosca   | 
| **Qual o meio de transporte mais usado nos trechos?** | Barras             | 



## Conexao com o banco 


In [ ]:
import sys
from pathlib import Path

# Script para arquivos notebook pegar a pasta atual do arquivo
raiz = Path().resolve().parent
if str(raiz) not in sys.path:
    sys.path.append(str(raiz))

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import banco

conexao = banco.conectar()

def consultar(sql):
    """Roda um SELECT e devolve o resultado como DataFrame do pandas."""
    return pd.read_sql(sql, conexao)

def reais(valor):
    """Formata um numero como moeda brasileira: 1234.5 -> 'R$ 1.234,50'."""
    texto = f'{valor:,.2f}'
    return 'R$ ' + texto.replace(',', 'X').replace('.', ',').replace('X', '.')

print('Conectado ao MySQL com sucesso.')

## Análise 1 – Os 5 órgãos com maior custo total
Realizado a soma total dos gastos com as viagens agrupado pelo órgão atrelado a viagem e ordenado de forma decrescente, para visualizar do custo maior ao menor.

O resultado está exibido em um gráfico de barras horizontal por facilitar a comparação entre categorias e acomodar nomes extensos dos órgãos.

In [ ]:
#Query SQL

SQL_ANALISE_1 = """
SELECT
    nome_orgao_superior,
    ROUND(SUM(valor_total),2) AS custo_total
FROM silver_viagem
GROUP BY nome_orgao_superior
ORDER BY custo_total DESC
LIMIT 5;
"""
dataframe_1 = consultar(SQL_ANALISE_1)
df1 = dataframe_1.copy()
df1["custo_total"] = df1["custo_total"].apply(reais)
display(df1)


#Plotagem do gráfico
plt.figure(figsize=(16,8))
plt.grid(axis='x', linestyle='--', alpha=0.4)

plt.barh(
    dataframe_1["nome_orgao_superior"],
    dataframe_1["custo_total"]
)

# Adiciona o valor ao final de cada barra
for i, valor in enumerate(dataframe_1["custo_total"]):
    plt.text(
        valor,                 # posição X
        i,                     # posição Y
        "  " + reais(valor),   # texto
        va='center',
        fontsize=10
    )

plt.title("TOP 5 Órgãos com maior custo total")
plt.xlabel("Custo total (R$)")
plt.ylabel("Órgão Superior")

plt.gca().invert_yaxis()

#Remove bordas dos eixos
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


# Desativa a notação científica (1e8)
plt.gca().ticklabel_format(style='plain', axis='x')

# Exibe o eixo em milhões
plt.gca().xaxis.set_major_formatter(
    FuncFormatter(lambda x, pos: f'{x/1_000_000:.0f} mi')
)

plt.tight_layout()
plt.show()



## Análise 2 – Qual UF de destino aparece em mais trechos?
A consulta contabiliza a quantidade de trechos de viagem por unidade federativa (UF), permitindo identificar quais estados concentram o maior número de deslocamentos registrados.
A tabela `silver_itens`, ja tem o `subtotal`, então foi agrupado por `categoria` e utilizado `SUM(subtotal)`.

O gráfico de rosca representa intuitivamente a participação de cada UF no total de trechos registrados. Sua visualização facilita a percepção da proporção de cada estado em relação ao conjunto dos dados, tornando a análise mais objetiva.

In [ ]:
#Query SQL
SQL_ANALISE_2 = """
SELECT
    destino_uf,
    COUNT(*) AS quantidade_trechos
FROM silver_trecho
GROUP BY destino_uf
ORDER BY quantidade_trechos DESC
LIMIT 10;
"""

dataframe_2 = consultar(SQL_ANALISE_2)
display(dataframe_2)

#Plotagem do gráfico
plt.figure(figsize=(8,8))

plt.pie(
    dataframe_2["quantidade_trechos"],
    labels=dataframe_2["destino_uf"],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'width':0.45}
)

plt.title("Participação das UFs nos trechos de viagem")

plt.show()

## Análise 3 – Qual o meio de transporte mais utilizado nos trechos?

Esta consulta contabiliza a quantidade de trechos realizados por cada meio de transporte, identificando qual modalidade é utilizada com maior frequência nas viagens oficiais.

O gráfico de barras verticais foi utilizado para comparar a frequência entre diferentes meios de transporte, permitindo identificar facilmente qual modalidade é predominante.

In [ ]:
#Query SQL
SQL_ANALISE_3 = """
SELECT
    meio_transporte,
    COUNT(*) AS quantidade
FROM silver_trecho
GROUP BY meio_transporte
ORDER BY quantidade DESC
LIMIT 5;
"""

dataframe_3 = consultar(SQL_ANALISE_3)
display(dataframe_3)

plt.figure(figsize=(10,6))

plt.bar(
    dataframe_3["meio_transporte"],
    dataframe_3["quantidade"]
)

plt.title("Meios de transporte mais utilizados")
plt.xlabel("Meio de transporte")
plt.ylabel("Quantidade")

plt.xticks(rotation=15)

plt.tight_layout()
plt.show()

--
## Camada GOLD Agregada

**Tabela GOLD agregada:** Construção de uma camada GOLD com JOIN + GROUP BY com fucoes de agregacao (`SUM`, `AVG`, `COUNT`, como tabela e como VIEW)

A tabelas agregadas foram criadas a partir das perguntas de negócio. Através delas é possível buscar informações de forma segmentada por assunto.

| Pergunta                                                          | Tipo de Gráfico                                                                             
| ----------------------------------------------------------------- | ----------------------- | 
| **Quais são os três destinos com maior custo médio por viagem?**  |xxxxxxx                  | 
| **A viagem de maior duração e seu custo total?**                  | xx                      | 
| **Qual tipo de pagamento possui o maior valor médio?**            | xxx                     | 


In [ ]:
# ==============================================================================
# LIMPEZA (Executar antes das criações)
# ==============================================================================

SQL_DROP_VIEWS = """
DROP VIEW IF EXISTS vw_gold_custo_orgao, 
                     vw_gold_resumo_destinos, 
                     vw_gold_duracao_viagens, 
                     vw_gold_uso_transportes, 
                     vw_gold_financeiro_pagamentos, 
                     vw_gold_frequencia_uf;
"""

SQL_DROP_TABELAS = """
DROP TABLE IF EXISTS gold_tb_custo_orgao, 
                     gold_tb_resumo_destinos, 
                     gold_tb_duracao_viagens, 
                     gold_tb_uso_transportes, 
                     gold_tb_financeiro_pagamentos, 
                     gold_tb_frequencia_uf;
"""

# ==============================================================================
# 1. Órgãos Superiores: Os 5 órgãos com maior custo total
# ==============================================================================

SQL_AGREGADA_1 = """
CREATE TABLE gold_tb_custo_orgao AS
SELECT 
    v.nome_orgao_superior,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens,
    SUM(v.valor_total) AS custo_total
FROM silver_viagem v
JOIN silver_pagamento p ON v.id_viagem = p.id_viagem
GROUP BY v.nome_orgao_superior;
"""

SQL_AGREGADA_VIEW_1 = """
CREATE VIEW vw_gold_custo_orgao AS SELECT * FROM gold_tb_custo_orgao;
"""

# ==============================================================================
# 2. Custo por Destino: Destinos com maior custo médio por viagem
# ==============================================================================

SQL_AGREGADA_2 = """
CREATE TABLE gold_tb_resumo_destinos AS
SELECT 
    t.destino_uf,
    t.destino_cidade,
    AVG(v.valor_total) AS custo_medio_viagem,
    COUNT(DISTINCT v.id_viagem) AS qtd_viagens_recebidas
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.destino_cidade IS NOT NULL
GROUP BY t.destino_uf, t.destino_cidade;
"""

SQL_AGREGADA_VIEW_2 = """
CREATE VIEW vw_gold_resumo_destinos AS SELECT * FROM gold_tb_resumo_destinos;
"""

# ==============================================================================
# 3. Duração e Escalas de Viagem: Viagem de maior duração e seu custo
# ==============================================================================

SQL_AGREGADA_3 = """
CREATE TABLE gold_tb_duracao_viagens AS
SELECT 
    v.id_viagem,
    v.nome_viajante,
    v.duracao_dias,
    v.valor_total AS custo_total_viagem,
    COUNT(t.id_trecho) AS total_trechos_na_viagem
FROM silver_viagem v
JOIN silver_trecho t ON v.id_viagem = t.id_viagem
GROUP BY v.id_viagem, v.nome_viajante, v.duracao_dias, v.valor_total;
"""

SQL_AGREGADA_VIEW_3 = """
CREATE VIEW vw_gold_duracao_viagens AS SELECT * FROM gold_tb_duracao_viagens;
"""


# ==============================================================================
# 4. Utilização de Transportes: Meio de transporte mais usado nos trechos
# ==============================================================================

SQL_AGREGADA_4 = """
CREATE TABLE gold_tb_uso_transportes AS
SELECT 
    t.meio_transporte,
    v.nome_orgao_superior,
    COUNT(t.id_trecho) AS qtd_trechos_percorridos
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
WHERE t.meio_transporte IS NOT NULL
GROUP BY t.meio_transporte, v.nome_orgao_superior;
"""

SQL_AGREGADA_VIEW_4 = """
CREATE VIEW vw_gold_uso_transportes AS SELECT * FROM gold_tb_uso_transportes;
"""

# ==============================================================================
# 5. Tipos de Pagamento: Tipo de pagamento com maior valor médio
# ==============================================================================

SQL_AGREGADA_5 = """
CREATE TABLE gold_tb_financeiro_pagamentos AS
SELECT 
    p.tipo_pagamento,
    COUNT(p.id_pagamento) AS qtd_pagamentos,
    AVG(p.valor) AS valor_medio_pagamento,
    SUM(p.valor) AS valor_total_pago
FROM silver_pagamento p
JOIN silver_viagem v ON p.id_viagem = v.id_viagem
GROUP BY p.tipo_pagamento;
"""

SQL_AGREGADA_VIEW_5 = """
CREATE VIEW vw_gold_financeiro_pagamentos AS SELECT * FROM gold_tb_financeiro_pagamentos;
"""


# ==============================================================================
# 6. Frequência Geográfica: UF de destino que aparece em mais trechos
# ==============================================================================

SQL_AGREGADA_6 = """
CREATE TABLE gold_tb_frequencia_uf AS
SELECT 
    t.destino_uf,
    COUNT(t.id_trecho) AS qtd_trechos_chegada,
    SUM(p.valor_passagem) AS gasto_total_passagens
FROM silver_trecho t
JOIN silver_viagem v ON t.id_viagem = v.id_viagem
LEFT JOIN silver_passagem p ON v.id_viagem = p.id_viagem
WHERE t.destino_uf IS NOT NULL
GROUP BY t.destino_uf;
"""

SQL_AGREGADA_VIEW_6 = """
CREATE VIEW vw_gold_frequencia_uf AS SELECT * FROM gold_tb_frequencia_uf;
"""

In [ ]:
# Função para execução das querys

def camada_gold():

    print("=== CRIAÇÃO DAS TABELAS CAMADA GOLD ===")

    try:

        conexao = banco.conectar()
        print(" ==== [1/3] Apagando tabelas caso já existam ====")
        banco.executar(conexao, SQL_DROP_TABELAS)
        print(" ==== [1/3] Tabelas apagadas OK ====")

        print(" ==== [2/3] Apagando views caso já existam ====")
        banco.executar(conexao, SQL_DROP_VIEWS)
        print(" ==== [2/3] Views apagadas OK ====")

        print(" ==== [3/3] Criando Tabelas e Views ====")
               
        banco.executar(conexao, SQL_AGREGADA_1)
        print("  Tabela gold_tb_analise_viagens OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_1)        
        print("  View vw_gold_analise_viagens OK")

        banco.executar(conexao, SQL_AGREGADA_2)
        print("  Tabela gold_tb_analise_logistica OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_2)        
        print("  View vw_gold_analise_logistica OK")

        banco.executar(conexao, SQL_AGREGADA_3)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_3)       
        print("  View vw_gold_analise_pagamentos OK")

        banco.executar(conexao, SQL_AGREGADA_4)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_4)       
        print("  View vw_gold_analise_pagamentos OK")

        banco.executar(conexao, SQL_AGREGADA_5)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_5)       
        print("  View vw_gold_analise_pagamentos OK")

        banco.executar(conexao, SQL_AGREGADA_6)
        print("  Tabela gold_tb_analise_pagamentos OK")
        banco.executar(conexao, SQL_AGREGADA_VIEW_6)       
        print("  View vw_gold_analise_pagamentos OK")

        conexao.close()

        print("=== CRIAÇÃO CAMADA GOLD AGREGADA CONCLUÍDA ===")

    except Exception as erro:

        print("[ERRO]", erro)
        raise


camada_gold()

## Análise 4: Quais são os três destinos com maior custo médio por viagem?
Gráficos de barras horizontais são perfeitos para exibir rankings

In [ ]:
SQL_ANALISE_4 = """
SELECT 
    destino_cidade, 
    destino_uf, 
    custo_medio_viagem
FROM vw_gold_resumo_destinos
ORDER BY custo_medio_viagem DESC
LIMIT 3;
"""

dataframe_4 = consultar(SQL_ANALISE_4)
display(dataframe_4)

# Cria identificadores
dataframe_4['destino_completo'] = dataframe_4['destino_cidade'] + " - " + dataframe_4['destino_uf']


fig, ax = plt.subplots(figsize=(9, 4.5))
# lista de cores (do azul escuro ao azul claro)
cores_degrade = ['#1e3a8a', '#3b82f6', '#93c5fd']

# ax.barh desenha as barras horizontais (Y = categorias, width = valores)
ax.barh(dataframe_4['destino_completo'], dataframe_4['custo_medio_viagem'], color=cores_degrade)

# Esse comando inverte o eixo Y 
ax.invert_yaxis()

# --- AJUSTES VISUAIS 

# Títulos e rótulos dos eixos
ax.set_title('Top 3 Destinos com Maior Custo Médio por Viagem', fontsize=14, pad=15, weight='bold')
ax.set_xlabel('Custo Médio (R$)', fontsize=11)
ax.set_ylabel('Destino', fontsize=11)

# Remove as linhas pretas da borda de cima e da direita (deixa o visual limpo)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Adiciona linhas de grade verticais e pontilhadas para ajudar a ler os valores
ax.grid(axis='x', linestyle='--', alpha=0.4)

# Exibe o gráfico na tela
plt.tight_layout()
plt.show()

## Análise 5: A viagem de maior duração e seu custo total?
Cartões de KPI Indicadores (Destaques Dinâmicos)

Cartão 1 (Principal): Número grande em destaque mostrando a quantidade de dias (Ex: 45 dias), com o nome do viajante em texto menor logo abaixo.

Cartão 2 (Secundário/Lado a Lado): Valor em Reais em destaque mostrando o custo total dessa mesma viagem (Ex: R$ 24.500,00).

In [ ]:
SQL_ANALISE_5 = """
SELECT 
    id_viagem, 
    nome_viajante, 
    duracao_dias, 
    custo_total_viagem
FROM vw_gold_duracao_viagens
ORDER BY duracao_dias DESC
LIMIT 1;
"""

dataframe_5 = consultar(SQL_ANALISE_5)
display(dataframe_5)

# Extrai os dados da única linha retornada pelo banco de forma dinâmica
if not dataframe_5.empty:
    viajante = dataframe_5['nome_viajante'].iloc[0]
    duracao_dias = dataframe_5['duracao_dias'].iloc[0]
    custo_total = dataframe_5['custo_total_viagem'].iloc[0]
else:
    viajante, duracao_dias, custo_total = "Sem registros", 0, 0.0

# 3. Alimenta e renderiza os cartões de destaque
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
fig.patch.set_facecolor('#f8fafc') 

# 1: Dias de Duração
ax[0].axis('off')
ax[0].text(0.5, 0.6, f"{duracao_dias} Dias", fontsize=32, weight='bold', ha='center', color='#1e3a8a')
ax[0].text(0.5, 0.3, f"Maior Duração Registrada\nViajante: {viajante}", fontsize=11, ha='center', color='#475569')

# 2: Custo Total atrelado
ax[1].axis('off')
ax[1].text(0.5, 0.6, f"R$ {custo_total:,.2f}", fontsize=28, weight='bold', ha='center', color='#b91c1c')
ax[1].text(0.5, 0.3, "Custo Total da Viagem", fontsize=11, ha='center', color='#475569')

plt.tight_layout()
plt.show()

## Análise 6: Qual tipo de pagamento possui o maior valor médio?
Consulta SQL

In [ ]:
SQL_ANALISE_6 = """
SELECT 
    tipo_pagamento, 
    valor_medio_pagamento
FROM vw_gold_financeiro_pagamentos
ORDER BY valor_medio_pagamento DESC
LIMIT 1;
"""
dataframe_6 = consultar(SQL_ANALISE_6)
display(dataframe_6)

# 3. O código descobre sozinho qual valor é o maior do banco para destacar
cor_destaque = '#1d4ed8'
cor_neutra = '#cbd5e1'
max_valor = dataframe_6['valor_medio_pagamento'].max()

# Varre os resultados aplicando a cor condicional
cores = [cor_destaque if v == max_valor else cor_neutra for v in dataframe_6['valor_medio_pagamento']]

# 4. Monta o gráfico com as colunas vindas do banco
plt.figure(figsize=(9, 5))
barras = plt.bar(dataframe_6['tipo_pagamento'], dataframe_6['valor_medio_pagamento'], color=cores)

# Insere dinamicamente os textos dos valores no topo de cada coluna
for barra in barras:
    altura = barra.get_height()
    plt.text(barra.get_x() + barra.get_width()/2, altura + (max_valor * 0.02), 
             f"R$ {altura:,.2f}", ha='center', va='bottom', fontsize=10, weight='bold')

# Configurações finais de exibição
plt.title('Valor Médio por Tipo de Pagamento', fontsize=14, pad=20, weight='bold')
plt.ylabel('Valor Médio (R$)', fontsize=11)
plt.xlabel('Método de Pagamento', fontsize=11)

# Alarga o topo do gráfico para o texto não sumir
plt.ylim(0, max_valor * 1.2) 

plt.tight_layout()
plt.show()

In [12]:
# Encerra a conexao com o banco
conexao.close()
print('Conexao encerrada.')

Conexao encerrada.
